# HW4: RAG Retrieval Evaluation

In this homework you'll wrap a retrieval pipeline and run it as a LangSmith experiment, pushing Recall@K and MRR metrics.

**What you'll learn:**
- How to upload datasets to LangSmith using the SDK
- How to wrap functions for use as LangSmith evaluators
- How to view and compare retrieval experiment results in LangSmith

## Setup

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. Load Recipes and Queries

`processed_recipes.json` contains 200 recipes. `synthetic_queries.jsonl` contains 200 queries, each linked to a source recipe via `source_recipe_id`.

In [2]:
import json

with open("processed_recipes.json") as f:
    recipes = json.load(f)

with open("synthetic_queries.jsonl") as f:
    queries = [json.loads(line) for line in f]

# Build a lookup from recipe id to recipe
recipe_by_id = {r["id"]: r for r in recipes}

print(f"Loaded {len(recipes)} recipes and {len(queries)} queries")
print(f"Example query: {queries[0]['query'][:80]}...")
print(f"Source recipe: {queries[0]['source_recipe_name']} (id={queries[0]['source_recipe_id']})")

Loaded 200 recipes and 200 queries
Example query: What temperature should I set my oven to and how long do I need to bake this swe...
Source recipe: amish friendship bread (id=246125)


## 2. Build a BM25 Retriever

Index each recipe's combined text (name + ingredients + steps) using BM25.

In [3]:
from rank_bm25 import BM25Okapi

# Build corpus: combine text fields for each recipe
corpus = []
recipe_ids = []

for r in recipes:
    text = f"{r['name']} {' '.join(r['ingredients'])} {' '.join(r['steps'])}"
    corpus.append(text.lower().split())
    recipe_ids.append(r["id"])

bm25 = BM25Okapi(corpus)


def retrieve(query: str, k: int = 5) -> list[int]:
    """Return top-k recipe IDs for a query using BM25."""
    scores = bm25.get_scores(query.lower().split())
    top_indices = scores.argsort()[::-1][:k]
    return [recipe_ids[i] for i in top_indices]


# Quick sanity check
test_results = retrieve(queries[0]["query"])
print(f"Query: {queries[0]['query'][:60]}...")
print(f"Expected: {queries[0]['source_recipe_id']}")
print(f"Retrieved: {test_results}")
print(f"Hit: {queries[0]['source_recipe_id'] in test_results}")

Query: What temperature should I set my oven to and how long do I n...
Expected: 246125
Retrieved: [139353, 246125, 338907, 478546, 181106]
Hit: True


## 3. Upload Dataset to LangSmith

Create a dataset where each example has the query as input and the source recipe ID as the expected output.

In [4]:
from langsmith import Client

client = Client()

DATASET_NAME = "recipe-retrieval"

dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description="BM25 retrieval evaluation — queries with known source recipe IDs.",
)

for q in queries:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"query": q["query"]},
        outputs={"source_recipe_id": q["source_recipe_id"]},
    )

print(f"Created dataset '{dataset.name}' with {len(queries)} examples.")

Created dataset 'recipe-retrieval' with 200 examples.


## 4. Define Retrieval Metrics

**Recall@K**: Is the source recipe in the top K results?
**MRR**: 1/rank of the source recipe (0 if not found).

In [5]:
def recall_at_k(run, example) -> dict:
    """Is the source recipe in the retrieved results?"""
    run_outputs = run.outputs if hasattr(run, "outputs") else run.get("outputs", {}) or {}
    example_outputs = example.outputs if hasattr(example, "outputs") else example.get("outputs", {}) or {}

    retrieved = run_outputs.get("retrieved_ids", [])
    expected = example_outputs.get("source_recipe_id")

    hit = expected in retrieved
    return {"score": 1.0 if hit else 0.0, "comment": f"{'Hit' if hit else 'Miss'} — expected {expected}"}


def mrr(run, example) -> dict:
    """Mean Reciprocal Rank: 1/rank of the source recipe."""
    run_outputs = run.outputs if hasattr(run, "outputs") else run.get("outputs", {}) or {}
    example_outputs = example.outputs if hasattr(example, "outputs") else example.get("outputs", {}) or {}

    retrieved = run_outputs.get("retrieved_ids", [])
    expected = example_outputs.get("source_recipe_id")

    for i, doc_id in enumerate(retrieved):
        if doc_id == expected:
            return {"score": 1.0 / (i + 1), "comment": f"Source recipe at rank {i + 1}."}

    return {"score": 0.0, "comment": "Source recipe not in results."}

## 5. Run Retrieval Experiments

Run two experiments: baseline (k=5) and expanded (k=10) to see if retrieving more docs improves recall.

In [6]:
from langsmith import evaluate


def run_retrieval_k5(inputs: dict) -> dict:
    return {"retrieved_ids": retrieve(inputs["query"], k=5)}


def run_retrieval_k10(inputs: dict) -> dict:
    return {"retrieved_ids": retrieve(inputs["query"], k=10)}


results_k5 = evaluate(
    run_retrieval_k5,
    data=DATASET_NAME,
    evaluators=[recall_at_k, mrr],
    experiment_prefix="retrieval-bm25-k5",
)

results_k10 = evaluate(
    run_retrieval_k10,
    data=DATASET_NAME,
    evaluators=[recall_at_k, mrr],
    experiment_prefix="retrieval-bm25-k10",
)

View the evaluation results for experiment: 'retrieval-bm25-k5-90031bc8' at:
https://smith.langchain.com/o/ebbaf2eb-769b-4505-aca2-d11de10372a4/datasets/dd733570-972b-41d9-8180-91e96950d583/compare?selectedSessions=196aa609-52b6-46f1-8ec5-4aa875caded7




0it [00:00, ?it/s]

View the evaluation results for experiment: 'retrieval-bm25-k10-794d7855' at:
https://smith.langchain.com/o/ebbaf2eb-769b-4505-aca2-d11de10372a4/datasets/dd733570-972b-41d9-8180-91e96950d583/compare?selectedSessions=41d09903-d2f3-4f38-9578-f9f140b99a29




0it [00:00, ?it/s]

## 6. View and Compare Results (UI)

### Steps:
1. Go to your dataset **recipe-retrieval** in LangSmith
2. Click the **Experiments** tab
3. Select both experiments (`retrieval-bm25-k5-*` and `retrieval-bm25-k10-*`) and click **Compare**

### What to look for:
- Does increasing K from 5 to 10 improve recall?
- Does MRR stay the same? (The rank of the first hit shouldn't change)
- Which query types have the worst retrieval performance?